# Create Materials Set

Save materials to a platform materials set — **ordered** (path order, e.g. NEB) or **unordered** (e.g. convex hull, EOS series).

This notebook only packages materials — it does not build or transform them. Build materials elsewhere (Materials Designer, a `create_*` notebook, or a dedicated builder notebook such as [`create_neb_path_materials.ipynb`](create_neb_path_materials.ipynb)), then either:

- Select them as **Input Materials** (outer runtime), or
- Write them to a subfolder under `uploads/` with `set_materials(materials, folder_path=f"uploads/{{SUBFOLDER_NAME}}")` — filename sort order becomes set order for ordered sets, so number material names (`00_...`, `01_...`, …).

## Usage

1. Set `MATERIAL_SET_NAME`, `IS_ORDERED`, and `SUBFOLDER_NAME` in cell 1.2.
1. Click "Run" > "Run All".

## Summary

1. Install packages and set parameters.
1. Authenticate and select account.
1. Load materials — from `SUBFOLDER_NAME` if set, otherwise Input Materials / the `uploads/` root.
1. Save materials on the platform and get or create the materials set (reuses `MATERIAL_SET_NAME` if it already exists, otherwise creates it).


## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)


In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")


### 1.2. Set parameters


In [ ]:
# Auth / organization
ORGANIZATION_NAME = None

# Materials set on the platform (use this name in workflow notebooks)
MATERIAL_SET_NAME = "NEB Silicon"
# True = path order via inSet.index (NEB); False = bag of members (hull, EOS, …)
IS_ORDERED = True

# Subfolder under uploads/ to load materials from (written there by a builder notebook via
# set_materials(materials, folder_path=f"uploads/{SUBFOLDER_NAME}")).
# None = use Input Materials (outer runtime), falling back to the uploads/ root.
SUBFOLDER_NAME = None


## 2. Authenticate and initialize API client
### 2.1. Authenticate


In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()


### 2.2. Initialize API client


In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client


### 2.3. Select account


In [ ]:
client.list_accounts()


In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")


## 3. Load materials for the set


In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize
from mat3ra.notebooks_utils.material import get_materials, load_materials_from_folder
from mat3ra.notebooks_utils.settings import UPLOADS_FOLDER

if SUBFOLDER_NAME:
    materials = load_materials_from_folder(f"{UPLOADS_FOLDER}/{SUBFOLDER_NAME}")
else:
    materials = get_materials()

print(f"Loaded {len(materials)} material(s): {[material.name for material in materials]}")
visualize(materials)


## 4. Save to a materials set
### 4.1. Create or reuse materials on the platform


In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = [get_or_create_material(client, material, ACCOUNT_ID) for material in materials]
for material, saved in zip(materials, saved_materials):
    print(f"{material.name}: {saved['_id']}")


### 4.2. Get or create the materials set and move members


In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_materials_set

materials_set = get_or_create_materials_set(
    client,
    ACCOUNT_ID,
    MATERIAL_SET_NAME,
    saved_materials,
    is_ordered=IS_ORDERED,
)
print(f"MATERIAL_SET = {materials_set['name']!r} (is_ordered={IS_ORDERED})")
